# Titanic - Machine Learning from Disaster

This is my first proper ML project. I'm using the Titanic dataset from Kaggle to practice the basic workflow: loading data, exploring it, cleaning it up, trying a couple of models, and making a submission.

I'm keeping things simple on purpose since I'm still learning - no fancy feature engineering or tuning, just the basics done carefully.

## 1. Load the data and libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

%matplotlib inline

In [ ]:
train = pd.read_csv('/kaggle/input/titanic/train.csv')
test = pd.read_csv('/kaggle/input/titanic/test.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)

*Note: I'm using the standard Kaggle input path here (`/kaggle/input/titanic/...`). If you're running this outside Kaggle, just point `pd.read_csv()` to wherever you saved train.csv and test.csv.*

## 2. First look at the data

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train.describe()

A few things I notice already:
- `Age` has missing values (count is less than 891)
- `Survived` is 0 or 1, so this is a binary classification problem
- `Fare` has a big range (min close to 0, max over 500), so there might be outliers

## 3. Checking missing values

In [ ]:
train.isnull().sum()

In [ ]:
test.isnull().sum()

So in `train.csv`:
- `Age` is missing a lot of values (177)
- `Cabin` is missing way too many values (687 out of 891) - I'm just going to drop this column, not worth trying to fill it in
- `Embarked` is only missing 2 values

In `test.csv`, `Age`, `Fare`, and `Cabin` have missing values too. I'll handle `Age` and `Fare` the same way I do for train.

## 4. A bit of EDA (exploratory data analysis)

Let's look at how survival relates to a few columns before building any model.

In [ ]:
sns.countplot(data=train, x='Survived')
plt.title('Survived (0 = No, 1 = Yes)')
plt.show()

In [ ]:
sns.countplot(data=train, x='Sex', hue='Survived')
plt.title('Survival count by Sex')
plt.show()

This one is pretty striking - a lot more women survived compared to men. So `Sex` is probably going to be an important feature.

In [ ]:
sns.countplot(data=train, x='Pclass', hue='Survived')
plt.title('Survival count by Passenger Class')
plt.show()

Passengers in 1st class survived at a much higher rate than 3rd class. Makes sense - probably had better access to lifeboats.

In [ ]:
train['Age'].hist(bins=30)
plt.title('Age distribution')
plt.xlabel('Age')
plt.show()

Most passengers are between roughly 20-40 years old, with a good number of kids too.

## 5. Handling missing values

My approach here is simple:
- `Age`: fill missing values with the **median** age (median is less sensitive to outliers than mean)
- `Embarked`: fill missing values with the **mode** (most common port), since only 2 values are missing
- `Fare` (in test set only): fill the 1 missing value with the median fare
- `Cabin`: dropping this column entirely, too many missing values to use it reliably

I calculate the median/mode from the **train** set only and use the same values to fill both train and test, so there's no data leakage from the test set.

In [ ]:
age_median = train['Age'].median()
train['Age'] = train['Age'].fillna(age_median)
test['Age'] = test['Age'].fillna(age_median)

embarked_mode = train['Embarked'].mode()[0]
train['Embarked'] = train['Embarked'].fillna(embarked_mode)

fare_median = train['Fare'].median()
test['Fare'] = test['Fare'].fillna(fare_median)

In [ ]:
# double check there's nothing left missing in the columns I actually plan to use
print(train[['Age', 'Embarked', 'Fare']].isnull().sum())
print(test[['Age', 'Embarked', 'Fare']].isnull().sum())

## 6. Picking features and encoding categorical columns

I'm keeping the feature list small and sticking to columns that are easy to reason about:
`Pclass`, `Sex`, `Age`, `SibSp`, `Parch`, `Fare`, `Embarked`.

I'm leaving out `Name`, `Ticket`, and `Cabin` - they'd need more work to turn into useful numeric features, and I want to keep this beginner-friendly.

`Sex` and `Embarked` are text columns, so I need to convert them to numbers before feeding them into a model.

In [ ]:
train['Sex'] = train['Sex'].map({'male': 0, 'female': 1})
test['Sex'] = test['Sex'].map({'male': 0, 'female': 1})

train['Embarked'] = train['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
test['Embarked'] = test['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

In [ ]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']

X = train[features]
y = train['Survived']

X_test_final = test[features]

X.head()

## 7. Train/validation split

I'm splitting the training data so I have a validation set to check how well each model does before touching the actual test set.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training rows:", X_train.shape[0])
print("Validation rows:", X_val.shape[0])

## 8. Trying two simple models

I'll try Logistic Regression first since it's a good simple baseline for classification, then Random Forest to see if it does better.

In [ ]:
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

log_preds = log_model.predict(X_val)
log_acc = accuracy_score(y_val, log_preds)

print("Logistic Regression validation accuracy:", round(log_acc, 4))

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_val)
rf_acc = accuracy_score(y_val, rf_preds)

print("Random Forest validation accuracy:", round(rf_acc, 4))

## 9. Comparing the two models

In [ ]:
print("Logistic Regression:", round(log_acc, 4))
print("Random Forest:", round(rf_acc, 4))

if rf_acc > log_acc:
    print("\nRandom Forest performed better, going with that.")
    final_model = rf_model
else:
    print("\nLogistic Regression performed better, going with that.")
    final_model = log_model

## 10. Training the final model on all the training data

Now that I've picked a model, I'll retrain it on the **full** training set (not just the 80% split) so it learns from as much data as possible before predicting on the test set.

In [ ]:
final_model.fit(X, y)

## 11. Predicting on the test set and creating submission.csv

In [ ]:
test_predictions = final_model.predict(X_test_final)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': test_predictions
})

submission.to_csv('submission.csv', index=False)
submission.head()

In [ ]:
print(submission.shape)
submission['Survived'].value_counts()

## 12. Final notes

- Validation accuracy was around 80% for Logistic Regression and a bit higher for Random Forest, so Random Forest was usually the one selected.
- This is a pretty basic approach - no hyperparameter tuning, no advanced feature engineering (like extracting titles from `Name` or family size from `SibSp`/`Parch`). Those would probably be my next steps if I wanted to improve the score.
- `Cabin`, `Name`, and `Ticket` were dropped entirely since they needed more preprocessing than I wanted to do for a first pass.

Overall this was a good exercise in the basic ML workflow: load data → explore → clean → encode → split → train → compare → predict → submit.